Microservices - REST (Synchron) - **Open Telemetry Variante**
---------------------------------------------------------

![](images/Microservices-REST.png)

Quelle: Buch Microservices Rezepte
- - -

Das Beispiel besteht aus drei Microservices: **Order**, **Customer** und **Catalog**. 

**Order** nutzt **Catalog** und **Customer** mit der REST-Schnittstelle. Ausserdem bietet jeder Microservice einige HTML-Seiten an.

Zusätzlich ist ein Service **Webshop** am laufen, der dem Benutzer mit einer Webseite einen einfachen Einstieg in das System ermöglicht.

- - -

Zuerst erstellen wir den Kubernetes Namespace ms-otel.

In [ ]:
! kubectl create namespace ms-otel

Jetzt ist ein guter Zeitpunkt um [HeadLamp](https://headlamp.dev/) (Nachfolger Kubernetes Dashboard) zu starten.

HeadLamp braucht einen Token welcher unter dem URL ausgegeben wird. Diesen markieren und mittels Ctrl+c kopieren und in der Headlamp Loginmaske wieder einfügen -> Ctrl+v.

In [ ]:
%%bash
echo "HeadLamp: http://"$(cat ~/data/server-ip)":30444"
kubectl create token headlamp-admin -n kube-system --duration=48h

Wir definieren eine OpenTelemetry-Instrumentation für den Namespace `ms-otel`, der die Python-Pods automatisch instrumentiert und ihre Traces per OTLP an den Collector sendet. 

Dabei werden Kontext-Propagation (W3C/B3) und Sampling konfiguriert, ohne dass Anpassungen am Anwendungscode nötig sind.D.h. die Services geben Trace-Informationen nach [W3C](https://www.w3.org/TR/trace-context/)/[B3](https://github.com/openzipkin/b3-propagation) weiter, damit Requests über mehrere Systeme verfolgbar sind.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: opentelemetry.io/v1alpha1
kind: Instrumentation
metadata:
  name: python-auto
  namespace: ms-otel
spec:
  exporter:
    endpoint: http://otel-collector-collector.opentelemetry.svc.cluster.local:4318

  propagators:
    - tracecontext
    - baggage
    - b3

  sampler:
    type: parentbased_traceidratio
    argument: "0.5"

  python:
    env:
      - name: OTEL_EXPORTER_OTLP_PROTOCOL
        value: http/protobuf
      - name: OTEL_TRACES_EXPORTER
        value: otlp
      - name: OTEL_METRICS_EXPORTER
        value: none
      - name: OTEL_LOGS_EXPORTER
        value: none
      - name: OTEL_RESOURCE_ATTRIBUTES
        value: service.namespace=ms-otel
EOF


Wir starten unsere Microservices.

Damit OpenTelemetry diese erkennt und automatisch instrumentiert wurde die Annotation `instrumentation.opentelemetry.io/inject-python` und `prometheus.io` hinzugefügt:

      template:
        metadata:
          labels:
            app: catalog
          annotations:
            instrumentation.opentelemetry.io/inject-python: "python-auto"
            prometheus.io/scrape: "true"
            prometheus.io/path: "/metrics"
            prometheus.io/port: "8080"

In [ ]:
%%bash
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-0-0-deployment/catalog-deployment.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-0-0-deployment/customer-deployment.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-0-0-deployment/order-deployment.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/3-0-0-deployment/webshop-deployment.yaml 
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/catalog-service.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/customer-service.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/order-service.yaml
kubectl apply --namespace ms-otel -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/webshop-service.yaml
kubectl get pod,services --namespace ms-otel

Da wir keinen LoadBalancer haben müssen wir mit einem kleinen Shellscript selber die IP des Clusters und der gemappte Port (port-based-routing) als URL aufbereiten.

In [ ]:
! echo "http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace ms-otel webshop -o=jsonpath='{ .spec.ports[0].nodePort }')/webshop

## OpenTelemetry Observability Stack (Prometheus, Grafana, Jaeger)

Prometheus ist das Metrics-Backend, Grafana ist die Oberfläche, und die OTel-Dashboards sind vorkonfigurierte Grafana-Dashboards für OpenTelemetry-/Service-Metriken. 

Traces laufen separat über den OpenTelemetry Collector nach Jaeger.


In [ ]:
%%bash
NAMESPACE="opentelemetry"
SERVER_IP="$(cat ~/data/server-ip 2>/dev/null || true)"
log() { echo $*; }

if [ -n "${SERVER_IP}" ]; then
  PROMETHEUS_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-prometheus -o=jsonpath='{.spec.ports[?(@.port==9090)].nodePort}')"
  ALERTMANAGER_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-alertmanager -o=jsonpath='{.spec.ports[?(@.port==9093)].nodePort}')"
  GRAFANA_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-grafana -o=jsonpath='{.spec.ports[?(@.port==80)].nodePort}')"
  JAEGER_PORT="$(kubectl -n "${NAMESPACE}" get svc jaeger -o=jsonpath='{.spec.ports[?(@.port==16686)].nodePort}')"

  log "Prometheus UI   : http://${SERVER_IP}:${PROMETHEUS_PORT}"
  log "Alertmanager UI : http://${SERVER_IP}:${ALERTMANAGER_PORT}"
  log "Grafana UI      : http://${SERVER_IP}:${GRAFANA_PORT}"
  log "Jaeger UI       : http://${SERVER_IP}:${JAEGER_PORT}"
else
  log "⚠️ [WARN] ~/data/server-ip nicht gefunden; NodePort-URLs nicht ausgegeben."
fi

### Lasttest

Um die Verbindungen sichtbar zu machen, erzeugen wir ein wenig Traffic.

Argumente
* -t 10s = Dauer in Sekunden
* -c 50  = Anzahl gleichzeitige Requests

In [ ]:
%%bash
sudo apt-get install -y apache2-utils -y
URL="http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace ms-otel webshop -o=jsonpath='{ .spec.ports[0].nodePort }')/webshop
ab -t 60s -c 10  ${URL}/order/order

- - -

Aufräumen

In [ ]:
! kubectl delete namespace ms-otel

- - -
### Quellen

* Sourcecode: https://gitlab.com/ch-mc-b/autoshop-ms/app/shop/-/tree/v3.0.0?ref_type=heads
* Kubernetes Deklarationen: https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates
* Container Registry: https://gitlab.com/ch-mc-b/autoshop-ms/app/shop/container_registry